# Gradients and Directional Derivatives

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 02.04 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/05_gradients_and_directional_derivatives.ipynb)

---

## 🎯 Learning Objective

Understand the gradient as the direction of steepest ascent, and directional derivatives as the rate of change along any chosen direction.

---

## 📐 Theory

In `02.03` the gradient $\nabla f = (\partial f/\partial x_1, \dots, \partial f/\partial x_n)$
was defined as a bundle of partial derivatives. This notebook asks: what does $\nabla f$ *mean*
geometrically, and how do you compute the rate of change in a direction that isn't one of the
coordinate axes?

### Directional derivatives

The **directional derivative** of $f$ at point $\mathbf{a}$ in the direction of a unit vector
$\hat{\mathbf{u}}$ is the rate of change of $f$ as you move from $\mathbf{a}$ along
$\hat{\mathbf{u}}$:

$$D_{\hat{\mathbf{u}}} f(\mathbf{a}) = \lim_{h\to 0}\frac{f(\mathbf{a}+h\hat{\mathbf{u}}) - f(\mathbf{a})}{h}$$

Partial derivatives are the special case where $\hat{\mathbf{u}}$ is a coordinate axis (e.g.
$\hat{\mathbf{u}}=(1,0,\dots,0)$ gives $\partial f/\partial x_1$). The key computational fact,
provable via the multivariate chain rule from `02.04` (treat $g(h) = f(\mathbf{a}+h\hat{\mathbf{u}})$
and differentiate at $h=0$), is:

$$D_{\hat{\mathbf{u}}} f(\mathbf{a}) = \nabla f(\mathbf{a}) \cdot \hat{\mathbf{u}}$$

The directional derivative in *any* direction is just the dot product of the gradient with
that direction. You never need a new formula per direction — the gradient already contains
enough information to answer the question for every direction at once.

### The gradient points toward steepest ascent

Recall from `01.01` that $\mathbf{a}\cdot\mathbf{b} = \|\mathbf{a}\|\|\mathbf{b}\|\cos\theta$.
Applying this to $D_{\hat{\mathbf{u}}}f(\mathbf{a}) = \nabla f(\mathbf{a})\cdot\hat{\mathbf{u}} =
\|\nabla f(\mathbf{a})\|\cos\theta$ (since $\hat{\mathbf{u}}$ has unit length), where $\theta$
is the angle between $\hat{\mathbf{u}}$ and $\nabla f(\mathbf{a})$. This is maximized exactly
when $\theta=0$ — i.e. when $\hat{\mathbf{u}}$ points in the *same direction* as
$\nabla f(\mathbf{a})$, giving a maximum rate of increase of $\|\nabla f(\mathbf{a})\|$. This
single piece of trigonometry is the entire justification for gradient-based optimization:

- $\nabla f(\mathbf{a})$ is the direction of **steepest ascent** (fastest increase).
- $-\nabla f(\mathbf{a})$ is the direction of **steepest descent** (fastest decrease).
- Moving perpendicular to $\nabla f(\mathbf{a})$ ($\theta=90°$) gives zero instantaneous
  change — this is the tangent direction to the level curve $f=c$ through $\mathbf{a}$,
  connecting back to the contour picture from `02.03`.

### Gradients are perpendicular to level curves

Since moving along a level curve doesn't change $f$, the directional derivative along the
level curve's tangent is $0$, which by $\nabla f\cdot\hat{\mathbf{u}}=0$ means $\nabla f$ is
*perpendicular* to that tangent. So $\nabla f(\mathbf{a})$ always points directly "outward,"
perpendicular to the contour line through $\mathbf{a}$, toward higher values of $f$ — this is
exactly the picture behind every 2D loss-landscape gradient arrow you'll see from here on.

### Why steepest descent, not "any decreasing direction"

Any direction with $\theta \in (90°, 270°)$ decreases $f$ locally, so gradient descent isn't
the *only* way to decrease a loss — but it decreases it *fastest*, per unit step size, which is
exactly the property that makes it a good default choice, revisited rigorously in Module `04`.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

On a contour plot, the gradient arrow at any point is perpendicular to the contour line
through that point and always points toward higher values — plotting gradient arrows across a
grid turns the whole surface's "flow toward the summit" into a single picture.

In [ ]:
# f(x,y) = x^2 + 2*y^2, an elliptical bowl. Gradient field overlaid on contours shows
# every arrow perpendicular to its local contour line, pointing "uphill".
f = lambda x, y: x**2 + 2 * y**2
grad_f = lambda x, y: np.array([2 * x, 4 * y])   # analytic gradient, by the power rule per-term

x_grid = np.linspace(-3, 3, 200)
y_grid = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f(X, Y)

# Sparser grid for the quiver arrows so they stay readable
xq, yq = np.meshgrid(np.linspace(-2.5, 2.5, 10), np.linspace(-2.5, 2.5, 10))
Uq, Vq = grad_f(xq, yq)
magnitude = np.sqrt(Uq**2 + Vq**2)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(X, Y, Z, levels=12, cmap="viridis", alpha=0.6)
ax.quiver(xq, yq, Uq / magnitude, Vq / magnitude, magnitude, cmap="autumn", scale=22)
a_pt = (1.5, 1.0)
ax.plot(*a_pt, 'ko', markersize=8)
g = grad_f(*a_pt)
ax.annotate("", xy=(a_pt[0] + g[0]*0.15, a_pt[1] + g[1]*0.15), xytext=a_pt,
            arrowprops=dict(arrowstyle="->", color="#C44E52", lw=2.5))
ax.set_title("Gradient field: every arrow perpendicular to its contour, pointing uphill")
ax.set_xlabel("x"); ax.set_ylabel("y")
plt.show()

print(f"At point {a_pt}: gradient = {g}, ||gradient|| = {np.linalg.norm(g):.3f}")
print("This is the direction of steepest ascent -- steepest DESCENT is exactly the negative of it.")

## 🐍 Implementation from Scratch

We compute directional derivatives directly from the definition (a 1D limit along a chosen
direction) and confirm the shortcut formula $D_{\hat{\mathbf{u}}}f = \nabla f\cdot\hat{\mathbf{u}}$,
then empirically confirm the gradient direction maximizes the directional derivative by scanning
every possible direction.

In [ ]:
def directional_derivative_numeric(f, point, direction, h=1e-6):
    """D_u f(a) straight from the definition: step h along the unit direction and
    measure the rate of change -- no dot-product shortcut used here."""
    direction = np.asarray(direction, dtype=float)
    norm_dir = np.linalg.norm(direction)
    if norm_dir == 0:
        # The zero vector has no direction to step along -- dividing by norm_dir=0 would
        # otherwise silently produce nan/inf. Raise instead, since "the directional
        # derivative along no direction" is not a meaningful question to ask.
        raise ValueError("direction must be nonzero -- the zero vector has no direction to differentiate along.")
    u = direction / norm_dir  # normalize to a unit vector
    point = np.array(point, dtype=float)
    return (f(*(point + h * u)) - f(*(point - h * u))) / (2 * h)

def gradient(f, point, h=1e-6):
    point = np.array(point, dtype=float)
    grad = np.zeros_like(point)
    for i in range(len(point)):
        step = np.zeros_like(point); step[i] = h
        grad[i] = (f(*(point + step)) - f(*(point - step))) / (2 * h)
    return grad

f = lambda x, y: x**2 + 2 * y**2
point = (1.5, 1.0)
grad_at_point = gradient(f, point)

# Check the shortcut formula D_u f = grad(f) . u_hat against 4 different directions
directions = {"+x axis": (1, 0), "+y axis": (0, 1), "diagonal": (1, 1), "gradient itself": tuple(grad_at_point)}
print(f"Gradient at {point}: {np.round(grad_at_point, 4)}\n")
for name, d in directions.items():
    u_hat = np.array(d) / np.linalg.norm(d)
    numeric = directional_derivative_numeric(f, point, d)
    shortcut = grad_at_point @ u_hat
    print(f"{name:16s} u_hat={np.round(u_hat,3)}  D_u(numeric)={numeric:.4f}  grad.u_hat={shortcut:.4f}")

# --- Regression test: a zero direction vector must raise a clear error, not return nan/inf ---
try:
    directional_derivative_numeric(f, point, (0, 0))
    print("\nFAILED to reject the zero direction vector (this should not happen)")
except ValueError as e:
    print(f"\nZero direction vector correctly rejected: {e}")

# Scan ALL directions (angle 0 to 360 degrees) and confirm the max directional derivative
# occurs exactly where theta aligns with the gradient's own direction
angles = np.linspace(0, 2*np.pi, 360)
directional_values = [grad_at_point @ np.array([np.cos(t), np.sin(t)]) for t in angles]
best_angle = angles[np.argmax(directional_values)]
gradient_angle = np.arctan2(grad_at_point[1], grad_at_point[0]) % (2*np.pi)
print(f"\nAngle that maximizes D_u f: {np.degrees(best_angle):.1f} deg")
print(f"Angle of the gradient itself:   {np.degrees(gradient_angle):.1f} deg  (should match)")

## 🤖 Why This Matters for AI

Gradient descent's update rule, $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla L(\mathbf{w})$,
is a direct application of this notebook's central result: $-\nabla L$ is the direction of
steepest *decrease*, so stepping along it shrinks the loss faster than stepping in any other
direction (for a small enough step size $\eta$, formalized in Module `04`). Directional
derivatives also power **feature attribution**: given a trained model, the directional
derivative of its output along a specific input direction tells you how sensitive the
prediction is to changing the input that way — the basis of saliency and "what-if" input
perturbation analysis used to interpret model decisions. Below, we implement one step of
gradient descent from first principles and verify it against the directional-derivative
argument: stepping along $-\nabla L$ decreases the loss more than stepping the same distance in
any other direction we try.

The specific techniques built on this gradient: **saliency maps** (Simonyan et al., 2014) visualize $|\nabla_{\text{input}} L|$ directly, **Integrated Gradients** (Sundararajan et al., 2017) integrate the gradient along a path from a baseline to the input, and **GradCAM** projects gradients onto convolutional feature maps. All three are available in Meta's `captum` library for PyTorch.

---


In [ ]:
# A toy "loss" L(w1, w2) for linear regression on one data point -- verify that stepping
# along -grad(L) beats stepping the same distance in any other direction.
x_data, y_data = 2.0, 5.0  # one training example: input x, target y
loss = lambda w1, w2: (w1 * x_data + w2 - y_data) ** 2

w_start = np.array([0.0, 0.0])
grad_L = gradient(loss, w_start)
step_size = 0.1

gradient_angle_deg = np.degrees(np.arctan2(grad_L[1], grad_L[0])) % 360
steepest_descent_angle = (gradient_angle_deg + 180) % 360
print(f"Gradient at w_start: {np.round(grad_L, 3)}  (points toward {gradient_angle_deg:.1f} deg)")
print(f"Loss at w_start: {loss(*w_start):.4f}\n")

# Fine sweep (every 2 degrees) so we can actually locate the true minimizing direction,
# rather than getting lucky/unlucky with a handful of coarse candidate directions
sweep_deg = np.arange(0, 360, 2)
sweep_losses = []
for angle_deg in sweep_deg:
    theta = np.radians(angle_deg)
    direction = np.array([np.cos(theta), np.sin(theta)])
    sweep_losses.append(loss(*(w_start + step_size * direction)))
sweep_losses = np.array(sweep_losses)
best_angle = sweep_deg[np.argmin(sweep_losses)]

print(f"Best direction found by sweep: {best_angle} deg")
print(f"-gradient direction (predicted optimum): {steepest_descent_angle:.1f} deg")
print(f"Difference: {abs(best_angle - steepest_descent_angle):.1f} deg (small, limited by the 2-degree sweep resolution)")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(sweep_deg, sweep_losses, color="#4C72B0", lw=2)
ax.axvline(steepest_descent_angle, color="#C44E52", ls="--", label="-gradient direction (predicted optimum)")
ax.set_xlabel("step direction (degrees)"); ax.set_ylabel("loss after one step")
ax.set_title("Steepest descent direction minimizes loss after one step")
ax.legend()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $f(x,y) = x^2 - xy$ at $(2,1)$, compute the directional derivative toward
   $\hat{\mathbf{u}} = (1,1)/\sqrt{2}$ both directly from the limit definition and via
   $\nabla f \cdot \hat{\mathbf{u}}$. As a sanity check, confirm $\hat{\mathbf{u}}$ is genuinely
   a unit vector before using it in the dot product.

<details>
<summary>💡 Solution</summary>

$\nabla f = (2x-y,\,-x)$, so $\nabla f(2,1) = (3,-2)$. Checking the direction is a unit vector
first: $\|(1,1)/\sqrt2\| = \sqrt{(1/\sqrt2)^2+(1/\sqrt2)^2} = \sqrt{1/2+1/2}=1$ ✓. Then
$D_{\hat{\mathbf{u}}}f(2,1) = \nabla f(2,1)\cdot\hat{\mathbf{u}} = (3)(1/\sqrt2)+(-2)(1/\sqrt2)
= 1/\sqrt2 \approx 0.7071$. `directional_derivative_numeric` (stepping directly along
$\hat{\mathbf{u}}$ and taking a central difference) reproduces $\approx0.70711$, matching the
dot-product shortcut to well within floating-point tolerance — confirming the shortcut formula
isn't an approximation, it's the exact same quantity computed a different way.

</details>

### 💭 UNDERSTAND
2. The Theory section claims $\nabla f(\mathbf{a})$ points toward *increasing* $f$, and
   $-\nabla f(\mathbf{a})$ toward *decreasing* $f$. A learner argues this can't always be
   right: "the gradient is just a vector built from partial derivatives — how does it 'know'
   which way is uphill?" Resolve this by explaining, from
   $D_{\hat{\mathbf{u}}}f = \|\nabla f\|\cos\theta$, precisely why the sign of the directional
   derivative is guaranteed to flip when you negate the direction, for *any* $f$ and any point
   where $\nabla f\neq\mathbf{0}$.

<details>
<summary>💡 Solution</summary>

The gradient doesn't need to "know" anything about the function's global shape — the
guarantee is a purely algebraic fact about the dot product, true regardless of what $f$ is.
$D_{\hat{\mathbf{u}}}f(\mathbf{a}) = \nabla f(\mathbf{a})\cdot\hat{\mathbf{u}} =
\|\nabla f(\mathbf{a})\|\cos\theta$, where $\theta$ is the angle to $\nabla f(\mathbf{a})$.
Negating $\hat{\mathbf{u}}$ replaces $\theta$ with $\theta+180°$, and $\cos(\theta+180°) =
-\cos\theta$ always — so $D_{-\hat{\mathbf{u}}}f(\mathbf{a}) = -D_{\hat{\mathbf{u}}}f(\mathbf{a})$
exactly, for every direction and every point (as long as $\nabla f(\mathbf{a})\ne\mathbf{0}$, so
$\theta$ is well-defined). Concretely, at $(2,1)$ on $f(x,y)=x^2-xy$ from COMPUTE, stepping a
small amount along $+\nabla f/\|\nabla f\|$ measurably *increases* $f$ (from $2.0$ to
$\approx2.0036$ over a step of $10^{-3}$) and the same-size step along $-\nabla f/\|\nabla f\|$
*decreases* it by the same amount to first order — this isn't a coincidence to verify case by
case, it's forced by the sign flip in $\cos\theta$.

</details>

### ✏️ DERIVE
3. Derive the shortcut formula $D_{\hat{\mathbf{u}}}f(\mathbf{a}) = \nabla f(\mathbf{a})\cdot
   \hat{\mathbf{u}}$ from the definition, for $f(x,y)=x^2+2y^2$ specifically: write
   $g(h)=f(\mathbf{a}+h\hat{\mathbf{u}})$ for $\mathbf{a}=(a_1,a_2)$,
   $\hat{\mathbf{u}}=(u_1,u_2)$, expand $g(h)$ as a polynomial in $h$, differentiate, and
   evaluate at $h=0$ to confirm it matches $\nabla f(\mathbf{a})\cdot\hat{\mathbf{u}}$ exactly.

<details>
<summary>💡 Solution outline</summary>

$g(h) = f(a_1+hu_1,\,a_2+hu_2) = (a_1+hu_1)^2 + 2(a_2+hu_2)^2$. Expanding:
$$g(h) = a_1^2+2a_1u_1h+u_1^2h^2 + 2a_2^2+4a_2u_2h+2u_2^2h^2
= \underbrace{(a_1^2+2a_2^2)}_{f(\mathbf{a})} + (2a_1u_1+4a_2u_2)h + (u_1^2+2u_2^2)h^2.$$
Differentiating in $h$: $g'(h) = (2a_1u_1+4a_2u_2) + 2(u_1^2+2u_2^2)h$, so
$$g'(0) = 2a_1u_1 + 4a_2u_2.$$
Separately, $\nabla f(\mathbf{a}) = (2a_1,4a_2)$, so $\nabla f(\mathbf{a})\cdot\hat{\mathbf{u}} =
2a_1u_1+4a_2u_2$ — identical to $g'(0)$. This is really just the single-variable chain rule
(`02.02`, `02.04`) applied to the composition $g(h)=f(\mathbf{a}+h\hat{\mathbf{u}})$: each
coordinate $a_i+hu_i$ is linear in $h$ with slope $u_i$, so $g'(0)=\sum_i
\frac{\partial f}{\partial x_i}(\mathbf{a})\cdot u_i$, which is exactly the dot product
$\nabla f(\mathbf{a})\cdot\hat{\mathbf{u}}$ written out.

</details>

### 🐍 IMPLEMENT
4. Write `find_steepest_direction_by_scan(f, point)` that finds the direction of steepest
   ascent by brute-force scanning many angles (e.g. 3600, at $0.1°$ resolution) and returning
   whichever unit vector gives the largest `directional_derivative_numeric`. Test it on a new,
   less symmetric function $f(x,y)=x^3-3xy^2+y^2$ at 2–3 points, and check the scan's answer
   against simply normalizing `gradient(f, point)`.

<details>
<summary>✅ How to know you're right</summary>

The scanned best direction and the normalized gradient should point the same way — cosine
similarity between them should be at least $0.999$ (limited only by the scan's angular
resolution; finer scanning pushes this even closer to exactly $1$), and the scan's best
directional-derivative *value* should closely match $\|\nabla f(\text{point})\|$ (the
predicted maximum rate of increase). If your scan's best direction disagrees noticeably with
the normalized gradient, check that `directional_derivative_numeric` is being called with a
properly normalized direction each time — an unnormalized direction vector changes the
directional derivative's magnitude (though not, for this exact formula, its sign or which angle
wins), which can subtly bias which angle the scan reports as best if the scan itself doesn't
normalize consistently.

</details>

### 🤖 APPLY
5. A simple linear classifier's confidence for class 1 on 2 input features is
   $\text{conf}(x_1,x_2) = \sigma(w_1x_1+w_2x_2)$ for fixed weights $\mathbf{w}$ (sigmoid
   $\sigma$). Given a starting input point and a small perturbation budget $\epsilon$, use the
   gradient to find the *direction* that increases the model's confidence the most, and compare
   it against 10,000 random unit-norm perturbation directions of the same budget.

<details>
<summary>✅ What you should observe</summary>

The confidence after stepping $\epsilon$ along the (normalized) gradient direction should be at
least as large as the confidence after stepping $\epsilon$ in *any* of the 10,000 random
directions tried — in one run, gradient-direction confidence reached $\approx0.6438$ versus a
best-of-10,000-random-directions confidence of $\approx0.6438$ as well (the random search
essentially rediscovers the same optimum by brute force, only because 10,000 dense directions
in 2D happens to get arbitrarily close) while the *average* random direction only reached
$\approx0.6270$, barely above the starting confidence of $0.6271$. This is precisely the
mathematical mechanism behind adversarial perturbations (e.g. the Fast Gradient Sign Method):
an attacker with query access to $\partial(\text{confidence})/\partial(\text{input})$ doesn't
need to search — the gradient directly identifies the most efficient direction to push an input
to change a model's prediction, for the same reason it identifies the most efficient direction
to decrease a loss during training.

</details>

### 🚀 CHALLENGE
6. When you can query a model's output but not its gradient directly (a black-box API, or a
   non-differentiable simulator), you can still *estimate* the gradient using only directional
   derivatives along many random directions: $\widehat{\nabla f}(\mathbf{a}) \approx
   \frac{d}{k}\sum_{i=1}^k D_{\hat{\mathbf{u}}_i}f(\mathbf{a})\,\hat{\mathbf{u}}_i$ for $k$
   random unit directions in $\mathbb{R}^d$. Investigate how the number of random directions
   $k$ needed to reach a good gradient estimate (say, cosine similarity $\geq0.95$ with the true
   gradient) scales with the dimension $d$, using a random quadratic test function at $d=2, 10,
   50, 100$.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer reports a roughly *linear* relationship between the dimension $d$ and the
number of random directions $k$ needed to reach cosine similarity $\geq0.95$ with the true
gradient: in one investigation, $d=2$ needed about $k=10$, $d=10$ needed about $k=100$, $d=50$
needed about $k=500$, and $d=100$ needed about $k=1000$ — consistently close to $k\approx10d$
across all four dimensions tested (averaging over repeated trials, since any single trial is
noisy — individual runs can vary quite a bit around this trend). A strong answer connects this
directly to the curse of dimensionality in zeroth-order optimization: each individual random
direction gives one honest scalar directional derivative, but a $d$-dimensional gradient has
$d$ independent numbers to pin down, so the number of queries needed to reconstruct it
accurately should be expected to grow with $d$ — this is exactly why gradient-free/black-box
optimization methods (evolution strategies, zeroth-order RL, some neural architecture search)
become dramatically more query-expensive as the number of parameters grows, and why, whenever
analytic gradients *are* available (as they are for any standard differentiable neural network
via backprop, `02.04`/`02.10`), using them is enormously more query-efficient than estimating
them this way — one backward pass gets the exact gradient in every one of the $d$ directions at
once, rather than needing on the order of $10d$ separate forward-pass queries for an approximate
one.

</details>

---

## 📚 Further Reading
- Deisenroth, Faisal & Ong, *Mathematics for Machine Learning*, Ch. 5.4–5.5 (Gradients, Directional Derivatives)
- Baydin et al., ["Automatic Differentiation in Machine Learning: a Survey"](https://arxiv.org/abs/1502.05767)

---

*Next notebook: [Jacobians and Hessians](06_jacobians_and_hessians.ipynb)*